# Alternative RCM BA - 2. Calibration of F2: RCM 2025 no hosing towards the reference quantiles

**Alternative solution** (to the supervisor's downscaling pipeline) for the bias adjustment of the RCM:

1. **F1** (`GCM_BA/`, done): WDF + QDM of EC-Earth3 towards ERA5 on the EC-Earth3 grid (0.7 deg), applied to
   `gcm_2025_nohosing` -> the target climate, ERA5-like, at 0.7 deg.
2. **Reference quantiles at 0.25 deg** (BCSD principle, Wood et al. 2004), per calendar month m (the 3 months centred on
   m, as F1) and quantile: Q_ref = Q_ERA5,0.25 + I[Q_F1 - Q_ERA5,0.7] for tasmax, ratios (interpolated in log) for DTR
   and the pr wet days; pr: dry-day fraction first, p0_ref = p0_ERA5,0.25 + I[p0_F1 - p0_ERA5,0.7] (ISIMIP3BASD
   principle, Lange 2019). I = bilinear interpolation from 0.7 to 0.25 deg. The RCM does not enter the reference.
3. **F2**: QDM of `rcm_2025_nohosing` towards Q_ref (`bu.fit_F2`: the xsdba training with Q_ref as ref quantiles),
   applied unchanged to the four RCM runs with `bu.apply_F` (the code of F1), each run ranked in its own distribution.

The RCM keeps its own day-to-day sequence and spatial structure (they carry the AMOC effect): the correction acts on
the distribution of each pixel only.

`bu.fit_F2`: per calendar month m, the xsdba training (`QuantileDeltaMapping.train`, 3-month window, the nodes of the
reference) on `rcm_2025_nohosing`, its adjustment factors computed against the reference quantiles
(`xsdba.utils.get_correction`): tasmax additive, DTR and pr multiplicative, pr dry days jittered below 0.1 mm/day as in
F1. The parameter files have the format of F1, so notebook 3 applies them with `bu.apply_F`.

In [ ]:
import os
os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import sys, json, time, warnings
from pathlib import Path
for _p in reversed((Path.cwd(), *Path.cwd().parents)):   # ba_utils.py (repository root): this folder or any parent folder
    sys.path.insert(0, str(_p))
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import ba_utils as bu
warnings.filterwarnings("ignore")

CONFIG_PATH = None   # None = config_<profile>.json found automatically (written by notebook 00)
USE_DASK_CLUSTER = False   # True = use several CPU cores (recommended on a JEO-lab / JEO-desk node)
cfg = bu.load_config(CONFIG_PATH)
CLIENT = bu.start_dask() if USE_DASK_CLUSTER else None
VARS = cfg["variables"]                       # pr, tasmax, tasmin (tasmin through DTR, as F1)

# ---- F1 (GCM_BA): the same tags as GCM_BA notebooks 2-3
CAL_PERIOD = ("1981", "2014")                 # calibration period of F1 (ERA5 / EC-Earth3 historical)
F1_TAG = f"F1_{CAL_PERIOD[0]}-{CAL_PERIOD[1]}_{bu.F1_VARIANT}_europe"
F1_PRODUCTS = bu.gcm_ba_dir(cfg, "GCM_bias_adjusted_0.7deg")

# ---- alternative solution
TARGET_GCM = "gcm_2025_nohosing"              # F1 product that gives the target climate of the RCM calibration
RCM_CAL = "rcm_2025_nohosing"                 # RCM run driven by it: F2 calibrated on it
RCM_RUNS = ["rcm_2025_nohosing", "rcm_2025_hosing", "rcm_2050_nohosing", "rcm_2050_hosing"]
MARGIN = 1.0                                  # degrees around the EFAS domain (LISFLOOD Europe)
REGION = {k: (bu.EFAS_BOX[k][0] - MARGIN, bu.EFAS_BOX[k][1] + MARGIN) for k in ("lat", "lon")}
COARSE_REGION = {k: (REGION[k][0] - 1.0, REGION[k][1] + 1.0) for k in ("lat", "lon")}   # + 1 coarse cell, for the interpolation
CHUNKS_SPACE = {"lat": 32, "lon": 64}
RCM_READY = bu.dataset_available(cfg, RCM_CAL, "pr")


def ref_years():
    """Years of the reference = years of the RCM calibration run (the F1 product over the same years); as long as the
    RCM is not in the config: all the years of the F1 product (to recompute once the RCM arrives)."""
    src = bu.read_raw(cfg, RCM_CAL, "pr") if RCM_READY else bu.read_processed(F1_PRODUCTS / f"{TARGET_GCM}_{F1_TAG}.nc", "pr")
    y = src.time.dt.year.values
    return str(y.min()), str(y.max())


REF_YEARS = ref_years()
QREF_TAG = f"{F1_TAG}_{TARGET_GCM}_{REF_YEARS[0]}-{REF_YEARS[1]}"
QREF_FILE = bu.alt_dir(cfg, "1_reference_quantiles") / f"qref_{QREF_TAG}.nc"
F2_DIR = bu.alt_dir(cfg, "2_F2_calculation", f"F2_{QREF_TAG}")
OUT = bu.alt_dir(cfg, "RCM_bias_adjusted_0.25deg")
print(f"region {REGION} | reference years {REF_YEARS[0]}-{REF_YEARS[1]}"
      + ("" if RCM_READY else " (RCM not in the config yet: all the years of the F1 product)"))
print("reference quantiles:", QREF_FILE, "\nF2:", F2_DIR, "\nproducts:", OUT)

## 1. Inputs: reference quantiles (notebook 1) and the RCM calibration run, same grid

In [ ]:
assert RCM_READY, f"{RCM_CAL}: RCM paths not in the config yet (notebook 00)"
assert QREF_FILE.exists(), f"{QREF_FILE.name} missing: run notebook 1 (after the RCM is in the config, for its years)"
qref = xr.open_dataset(QREF_FILE).load()
hist = {v: bu.subset_time(bu.crop_bbox(bu.read_raw(cfg, RCM_CAL, v), REGION), *REF_YEARS) for v in VARS}
g = hist["pr"]
if not bu.same_grid(g, qref):
    raise ValueError(f"RCM grid ({g.sizes['lat']} x {g.sizes['lon']}, {float(g.lat[0]):.3f}..{float(g.lat[-1]):.3f} N) is not the "
                     f"ERA5 0.25 deg grid of the reference ({qref.sizes['lat']} x {qref.sizes['lon']}): decide how to bring the "
                     "reference onto the RCM grid (or the RCM onto ERA5's) before F2")
print(f"{RCM_CAL}: {g.sizes['time']} days ({REF_YEARS[0]}-{REF_YEARS[1]}), {g.sizes['lat']} x {g.sizes['lon']} pixels")

## 2. Calibration

In [ ]:
OVERWRITE = False
t0 = time.time()
bu.fit_F2(qref, hist, F2_DIR, chunks_space=CHUNKS_SPACE, overwrite=OVERWRITE)
(F2_DIR / "f2_info.json").write_text(json.dumps({"reference": QREF_FILE.name, "rcm": RCM_CAL, "years": REF_YEARS,
                                                 "window_months": int(qref.attrs["window_months"]), "pr_method": "jitter",
                                                 "nquantiles": int(qref.attrs["nquantiles"])}, indent=2))
print(f"F2 calibration: {(time.time() - t0) / 60:.1f} min |", sorted(p.name for p in F2_DIR.iterdir()))

## 3. In-sample check (first 3 years)

F2 applied to `rcm_2025_nohosing` itself: its distribution should fall back onto the reference (code check; the
real checks are in notebook 4).

In [ ]:
sl = slice(0, 365 * 3)
H = {v: x.isel(time=sl) for v, x in hist.items()}
A = xr.Dataset(bu.apply_F(H, F2_DIR, chunks_space=CHUNKS_SPACE)).compute()
rows = []
for v in [v for v in VARS if v in A]:
    for lab, x in (("raw RCM", H[v].compute()), ("RCM + F2", A[v])):
        s = bu.summarize(x)
        rows.append({"var": v, "series": lab, **{k: round(val, 3) for k, val in s.items() if k != "n"}})
display(pd.DataFrame(rows).set_index(["var", "series"]))